# M2 Gap — ARIMA Univariate Baseline

**Issue:** #54  
**Owner:** Mitchel  
**Reviewer:** Nelson  
**Branch:** `artifact/m2-arima-baseline`

## Objective

Implement the univariate ARIMA statistical baseline required by the project proposal and compare it directly against the shared Random Walk benchmark used in the VAR baselines.

The model will use the stationary Canadian 10Y–2Y yield-spread series and will follow the same evaluation framework used in Issues #28 and #29 to preserve direct comparability across models.

## Methodology

- Target: `yield_spread_10y_2y`
- Stationarity treatment: first-differenced spread based on the Round 2 ADF findings
- ARIMA order selection: automated search minimizing AIC/BIC
- Forecast horizons: 1, 5, and 20 observations
- Expanding-window evaluation
- Initial training sample: 500 observations
- Evaluation step: 5 observations
- Benchmark: Random Walk
- Metrics: RMSE and MAE
- Statistical comparison: Diebold-Mariano test using the same methodology as #28/#29

## Comparability requirement

The ARIMA baseline must reproduce from the same processed data and evaluation window used by the corrected Round 3 VAR baselines so that ARIMA, VAR-BIC, VAR-AIC, and the Random Walk can be compared directly.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.arima.model import ARIMA

# 296 non-convergent fits (263 arima_aic + 33 arima_bic) across the
# expanding-window loop below print a UserWarning + ConvergenceWarning
# each -- both include the full local
# filesystem path to the statsmodels install in the traceback, which leaks
# into this notebook's committed output on every contributor's machine
# (issue #74). Suppressed the same way src/johansen_vecm.py already
# suppresses its own statsmodels ValueWarning -- scoped to statsmodels only,
# not a blanket warnings.filterwarnings("ignore"). Whether non-convergent
# fits should be excluded from the reported metrics is issue #71's separate,
# substantive question; this only stops the traceback from being printed.
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=ConvergenceWarning, module="statsmodels")

import sys

# Bootstrap guess, just precise enough to import project_paths -- immediately
# replaced below by the authoritative, marker-based find_project_root(), so a
# wrong guess here (e.g. this notebook run from a different working directory)
# doesn't silently propagate into every downstream path.
_bootstrap_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_bootstrap_root / "src"))
from project_paths import find_project_root
PROJECT_ROOT = find_project_root()
PROCESSED = PROJECT_ROOT / "data" / "processed"

TARGET = "yield_spread_10y_2y"
HORIZONS = [1, 5, 20]

MIN_TRAIN = 500
STEP = 5

print("Target:", TARGET)
print("Horizons:", HORIZONS)

Target: yield_spread_10y_2y
Horizons: [1, 5, 20]


In [2]:
import sys

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from model_features import ROUND3_FEATURES

In [3]:
# Load Bank of Canada processed data
boc = pd.read_csv(
    PROCESSED / "bank_of_canada_data.csv",
    parse_dates=["date"]
)

spread_levels = (
    boc[["date", TARGET]]
    .dropna(subset=[TARGET])
    .sort_values("date")
    .reset_index(drop=True)
)

spread_diff = spread_levels.copy()
spread_diff[TARGET] = spread_diff[TARGET].diff()

spread_diff = (
    spread_diff
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

print("Level observations:", len(spread_levels))
print("Differenced observations:", len(spread_diff))
print(
    "Date range:",
    spread_diff["date"].min(),
    "->",
    spread_diff["date"].max()
)

spread_diff.head()

Level observations: 4368
Differenced observations: 4367
Date range: 2009-01-05 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y
0,2009-01-05,0.06
1,2009-01-06,-0.01
2,2009-01-07,0.06
3,2009-01-08,-0.06
4,2009-01-09,-0.01


In [4]:
# Build the same common evaluation sample used by #28/#29 -- issue #63: reads the
# canonical Gold-layer CSV directly instead of independently rebuilding an
# inner-joined, no-fill daily frame that silently disagreed with VECM/LSTM's
# calendar (see src/EDA_VAR_AIC _lag _order.py's load_levels() for the same fix).
daily = pd.read_csv(
    PROCESSED / "gold_features.csv",
    parse_dates=["date"]
)

COMMON_FEATURES = ROUND3_FEATURES

# Drop incomplete level observations BEFORE differencing,
# exactly as in the corrected VAR baseline
common_levels = (
    daily[["date"] + COMMON_FEATURES]
    .dropna(subset=COMMON_FEATURES)
    .sort_values("date")
    .reset_index(drop=True)
)

# ARIMA remains univariate: retain only the spread after date alignment
arima_levels = common_levels[["date", TARGET]].copy()

arima_diff = arima_levels.copy()
arima_diff[TARGET] = arima_diff[TARGET].diff()

arima_diff = (
    arima_diff
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

print("Common level observations:", len(arima_levels))
print("ARIMA differenced observations:", len(arima_diff))
print(
    "ARIMA modeling date range:",
    arima_diff["date"].min(),
    "->",
    arima_diff["date"].max()
)

Common level observations: 4268
ARIMA differenced observations: 4267
ARIMA modeling date range: 2010-02-22 00:00:00 -> 2026-06-30 00:00:00


In [5]:
# Automated ARIMA order search on the stationary spread series

stationary_spread = arima_diff[TARGET].astype(float)

MAX_P = 5
MAX_Q = 5

order_search_results = []

for p in range(MAX_P + 1):
    for q in range(MAX_Q + 1):
        order = (p, 0, q)

        try:
            fitted = ARIMA(
                stationary_spread,
                order=order,
                trend="c"
            ).fit()

            order_search_results.append({
                "p": p,
                "d": 0,
                "q": q,
                "aic": fitted.aic,
                "bic": fitted.bic,
            })

        except Exception as exc:
            print(f"Skipped ARIMA{order}: {exc}")

order_search_df = (
    pd.DataFrame(order_search_results)
    .sort_values("aic")
    .reset_index(drop=True)
)

best_aic_row = order_search_df.loc[
    order_search_df["aic"].idxmin()
]

best_bic_row = order_search_df.loc[
    order_search_df["bic"].idxmin()
]

BEST_AIC_ORDER = (
    int(best_aic_row["p"]),
    0,
    int(best_aic_row["q"])
)

BEST_BIC_ORDER = (
    int(best_bic_row["p"]),
    0,
    int(best_bic_row["q"])
)

print("Best order by AIC:", BEST_AIC_ORDER)
print("AIC:", round(best_aic_row["aic"], 3))

print("\nBest order by BIC:", BEST_BIC_ORDER)
print("BIC:", round(best_bic_row["bic"], 3))

print("\nTop 10 models by AIC:")
display(order_search_df.head(10))

Best order by AIC: (2, 0, 3)
AIC: -18143.9

Best order by BIC: (0, 0, 0)
BIC: -18118.356

Top 10 models by AIC:


,p,d,q,aic,bic
0,2,0,3,-18143.899637,-18099.388973
1,3,0,4,-18138.729691,-18081.501694
2,4,0,3,-18138.420862,-18081.192866
3,5,0,0,-18137.098233,-18092.587569
4,0,0,5,-18136.960417,-18092.449753
5,1,0,4,-18135.949279,-18091.438615
6,0,0,4,-18135.710735,-18097.558737
7,2,0,4,-18135.296325,-18084.426994
8,4,0,0,-18135.193415,-18097.041418
9,5,0,1,-18135.127616,-18084.258285


In [6]:
# Validate convergence of the AIC- and BIC-selected specifications

selected_models = {}

for criterion, order in {
    "AIC": BEST_AIC_ORDER,
    "BIC": BEST_BIC_ORDER,
}.items():

    fit = ARIMA(
        stationary_spread,
        order=order,
        trend="c"
    ).fit()

    selected_models[criterion] = fit

    converged = fit.mle_retvals.get("converged", True)

    print(f"{criterion}-selected model: ARIMA{order}")
    print(f"Converged: {converged}")
    print(f"AIC: {fit.aic:.3f}")
    print(f"BIC: {fit.bic:.3f}")
    print("-" * 40)

AIC-selected model: ARIMA(2, 0, 3)
Converged: True
AIC: -18143.900
BIC: -18099.389
----------------------------------------


BIC-selected model: ARIMA(0, 0, 0)
Converged: True
AIC: -18131.073
BIC: -18118.356
----------------------------------------


In [7]:
# Issue #63 follow-up: origin is now counted in LEVEL observations trained on
# (train = arima_diff.iloc[:origin - 1], last_level/origin_date = arima_levels.iloc[origin - 1]),
# matching evaluate_vecm()'s convention exactly instead of counting DIFFERENCED
# observations trained on -- the previous aligned_arima_levels.loc[origin - 1] anchored
# last_level one row later than VECM's origin - 1 for the same `origin` number, a
# permanent one-row phase offset that desynced this notebook's origin grid from VECM's
# even on an identical calendar (0 of 750x750 origin_date pairs coincided). Confirms the
# index relationship the loop below relies on: arima_diff.iloc[i] is the change ending on
# arima_levels.iloc[i + 1], so training on `origin` levels uses `origin - 1` diffs.
assert len(arima_levels) == len(arima_diff) + 1
assert (arima_diff["date"].reset_index(drop=True) == arima_levels["date"].iloc[1:].reset_index(drop=True)).all()

print("Level observations:", len(arima_levels))
print("Differenced observations:", len(arima_diff))

Level observations: 4268
Differenced observations: 4267


In [8]:
ARIMA_ORDERS = {
    "arima_aic": BEST_AIC_ORDER,
    "arima_bic": BEST_BIC_ORDER,
}

results = []

for origin in range(MIN_TRAIN, len(arima_levels) - max(HORIZONS), STEP):
    train_diff = arima_diff.iloc[:origin - 1][TARGET].astype(float)

    # Last observed level at forecast origin
    last_level = arima_levels.loc[origin - 1, TARGET]

    fitted_models = {}
    converged_flags = {}

    for model_name, order in ARIMA_ORDERS.items():
        fit = ARIMA(
            train_diff,
            order=order,
            trend="c"
        ).fit()
        fitted_models[model_name] = fit
        # Captured here, at the point of fitting, rather than in a later cell
        # that used to re-fit every origin a second time just to read this
        # flag (issue #71) -- also removes that duplicate ~1,400-fit pass.
        converged_flags[model_name] = bool(fit.mle_retvals.get("converged", True))

    for h in HORIZONS:
        actual_level = arima_levels.loc[origin + h - 1, TARGET]

        # Shared Random Walk benchmark
        naive_forecast = last_level

        row = {
            "origin_date": arima_levels.loc[origin - 1, "date"],
            "origin_idx": origin,
            "horizon": h,
            "actual": actual_level,
            "naive": naive_forecast,
        }

        for model_name, fit in fitted_models.items():
            forecast_diff = np.asarray(
                fit.forecast(steps=h),
                dtype=float
            )

            cumulative_change = forecast_diff.sum()
            row[model_name] = last_level + cumulative_change
            row[f"{model_name}_converged"] = converged_flags[model_name]

        results.append(row)

results_df = pd.DataFrame(results)

print("Forecast rows:", len(results_df))
print(results_df.groupby("horizon").size())

results_df.head()

Forecast rows: 2250
horizon
1     750
5     750
20    750
dtype: int64


,origin_date,origin_idx,horizon,actual,naive,arima_aic,arima_aic_converged,arima_bic,arima_bic_converged
0,2012-01-19,500,1,1.02,0.98,0.976010,False,0.977730,True
1,2012-01-19,500,5,1.01,0.98,0.969677,False,0.968652,True
2,2012-01-19,500,20,0.96,0.98,0.941134,False,0.934610,True
3,2012-01-26,505,1,0.99,1.01,1.011247,False,1.007812,True
4,2012-01-26,505,5,0.95,1.01,1.003523,False,0.999062,True


In [9]:
print("Total forecast rows:", len(results_df))

print("\nMissing forecasts:")
print(
    results_df[
        ["arima_aic", "arima_bic", "naive", "actual"]
    ].isna().sum()
)

print("\nInfinite forecasts:")
print(
    np.isinf(
        results_df[
            ["arima_aic", "arima_bic", "naive", "actual"]
        ]
    ).sum()
)

print("\nForecast ranges:")
print(
    results_df[
        ["actual", "naive", "arima_aic", "arima_bic"]
    ].agg(["min", "max"])
)

Total forecast rows: 2250

Missing forecasts:
arima_aic    0
arima_bic    0
naive        0
actual       0
dtype: int64

Infinite forecasts:
arima_aic    0
arima_bic    0
naive        0
actual       0
dtype: int64

Forecast ranges:
     actual  naive  arima_aic  arima_bic
min   -1.32  -1.30  -1.322994  -1.319675
max    1.64   1.61   1.607364   1.609499


In [10]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]

    row = {
        "horizon": h
    }

    for model_name in ["arima_aic", "arima_bic", "naive"]:
        row[f"{model_name}_rmse"] = np.sqrt(
            mean_squared_error(
                subset["actual"],
                subset[model_name]
            )
        )

        row[f"{model_name}_mae"] = mean_absolute_error(
            subset["actual"],
            subset[model_name]
        )

    # Improvement relative to Random Walk
    for model_name in ["arima_aic", "arima_bic"]:
        row[f"{model_name}_rmse_improvement_pct"] = (
            (row["naive_rmse"] - row[f"{model_name}_rmse"])
            / row["naive_rmse"]
            * 100
        )

        row[f"{model_name}_mae_improvement_pct"] = (
            (row["naive_mae"] - row[f"{model_name}_mae"])
            / row["naive_mae"]
            * 100
        )

    metrics.append(row)

metrics_df = pd.DataFrame(metrics)

metrics_df.round(6)

,horizon,arima_aic_rmse,arima_aic_mae,arima_bic_rmse,arima_bic_mae,naive_rmse,naive_mae,arima_aic_rmse_improvement_pct,arima_aic_mae_improvement_pct,arima_bic_rmse_improvement_pct,arima_bic_mae_improvement_pct
0,1,0.029104,0.020794,0.029010,0.020665,0.029001,0.020587,-0.355337,-1.009312,-0.030587,-0.378723
1,5,0.061872,0.047193,0.061355,0.046630,0.061161,0.046333,-1.162796,-1.856169,-0.317237,-0.640095
2,20,0.128153,0.096310,0.126546,0.095691,0.125057,0.094387,-2.475300,-2.037641,-1.190476,-1.382095


### Convergence-conditional metrics (issue #71)

The table above scores every expanding-window origin regardless of whether that origin's MLE fit
converged (64.9%/95.6% convergence rate for AIC/BIC, per the diagnostic below). That means
forecasts and reported accuracy metrics may include predictions from fits that never converged
-- which don't carry the asymptotic validity the rest of this evaluation assumes.

Reported side by side rather than silently dropped: **all_origins** (identical to the table
above, kept for continuity) and **converged_only** (the same metric, computed only on origins
where that specific model's fit reported `converged=True`; Naive is recomputed on the same
matched subset for a fair comparison, since it's a different subset per model).

In [11]:
def _metrics_row(actual, model_pred, naive_pred):
    rmse = float(np.sqrt(mean_squared_error(actual, model_pred)))
    mae = float(mean_absolute_error(actual, model_pred))
    naive_rmse = float(np.sqrt(mean_squared_error(actual, naive_pred)))
    naive_mae = float(mean_absolute_error(actual, naive_pred))
    return {
        "n_forecasts": len(actual),
        "rmse": rmse, "mae": mae,
        "naive_rmse": naive_rmse, "naive_mae": naive_mae,
        "rmse_improvement_pct": (naive_rmse - rmse) / naive_rmse * 100,
        "mae_improvement_pct": (naive_mae - mae) / naive_mae * 100,
    }


convergence_metric_rows = []
for h in HORIZONS:
    subset_all = results_df[results_df["horizon"] == h]
    for model_name in ["arima_aic", "arima_bic"]:
        r_all = _metrics_row(subset_all["actual"], subset_all[model_name], subset_all["naive"])
        r_all.update({"horizon": h, "model": model_name, "sample": "all_origins"})
        convergence_metric_rows.append(r_all)

        conv_col = f"{model_name}_converged"
        subset_conv = subset_all[subset_all[conv_col]]
        r_conv = _metrics_row(subset_conv["actual"], subset_conv[model_name], subset_conv["naive"])
        r_conv.update({"horizon": h, "model": model_name, "sample": "converged_only"})
        convergence_metric_rows.append(r_conv)

metrics_by_convergence_df = pd.DataFrame(convergence_metric_rows)[
    ["horizon", "model", "sample", "n_forecasts", "rmse", "mae",
     "naive_rmse", "naive_mae", "rmse_improvement_pct", "mae_improvement_pct"]
].sort_values(["horizon", "model", "sample"]).reset_index(drop=True)

metrics_by_convergence_df.round(6)

,horizon,model,sample,n_forecasts,rmse,mae,naive_rmse,naive_mae,rmse_improvement_pct,mae_improvement_pct
0,1,arima_aic,all_origins,750,0.029104,0.020794,0.029001,0.020587,-0.355337,-1.009312
1,1,arima_aic,converged_only,487,0.029764,0.021118,0.029697,0.020924,-0.224273,-0.929059
2,1,arima_bic,all_origins,750,0.029010,0.020665,0.029001,0.020587,-0.030587,-0.378723
3,1,arima_bic,converged_only,717,0.028667,0.020651,0.028657,0.020558,-0.035578,-0.452538
4,5,arima_aic,all_origins,750,0.061872,0.047193,0.061161,0.046333,-1.162796,-1.856169
5,5,arima_aic,converged_only,487,0.064510,0.049420,0.064068,0.048665,-0.690162,-1.550213
6,5,arima_bic,all_origins,750,0.061355,0.046630,0.061161,0.046333,-0.317237,-0.640095
7,5,arima_bic,converged_only,717,0.060989,0.046410,0.060828,0.046151,-0.265463,-0.562369
8,20,arima_aic,all_origins,750,0.128153,0.096310,0.125057,0.094387,-2.475300,-2.037641
9,20,arima_aic,converged_only,487,0.134661,0.100353,0.132358,0.098398,-1.739970,-1.986615


In [12]:
# Diebold-Mariano tests: ARIMA-AIC vs Naive and ARIMA-BIC vs Naive.
# Run on both samples (issue #71) -- all_origins (previous behavior, unchanged)
# and converged_only (excludes origins where that model's fit didn't converge),
# same matched-subset logic as the metrics table above.

from dieboldmariano import dm_test

dm_results = []

for h in HORIZONS:
    subset_all = results_df[results_df["horizon"] == h].copy()

    for model_name in ["arima_aic", "arima_bic"]:
        for sample_name, subset in [
            ("all_origins", subset_all),
            ("converged_only", subset_all[subset_all[f"{model_name}_converged"]]),
        ]:
            actual = subset["actual"].to_numpy()
            naive = subset["naive"].to_numpy()
            model_fcst = subset[model_name].to_numpy()

            # Squared-error loss
            dm_stat_sq, p_sq = dm_test(
                actual,
                model_fcst,
                naive,
                h=h,
                one_sided=False,
                harvey_correction=True,
                variance_estimator="bartlett"
            )

            # Absolute-error loss
            dm_stat_abs, p_abs = dm_test(
                actual,
                model_fcst,
                naive,
                h=h,
                loss=lambda u, v: abs(u - v),
                one_sided=False,
                harvey_correction=True,
                variance_estimator="bartlett"
            )

            dm_results.append({
                "model": model_name,
                "horizon_days": h,
                "sample": sample_name,
                "n_forecasts": len(subset),
                "dm_stat_squared_loss": dm_stat_sq,
                "dm_p_value_squared_loss": p_sq,
                "dm_stat_absolute_loss": dm_stat_abs,
                "dm_p_value_absolute_loss": p_abs,
            })

dm_results_df = pd.DataFrame(dm_results)

dm_results_df.round(4)

,model,horizon_days,sample,n_forecasts,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss
0,arima_aic,1,all_origins,750,1.2137,0.2253,2.4137,0.0160
1,arima_aic,1,converged_only,487,0.6711,0.5025,1.9593,0.0507
2,arima_bic,1,all_origins,750,0.2657,0.7905,2.2626,0.0239
3,arima_bic,1,converged_only,717,0.2957,0.7676,2.6395,0.0085
4,arima_aic,5,all_origins,750,1.9956,0.0463,2.1923,0.0287
5,arima_aic,5,converged_only,487,0.9603,0.3374,1.4317,0.1529
6,arima_bic,5,all_origins,750,1.2085,0.2272,1.7145,0.0869
7,arima_bic,5,converged_only,717,0.9516,0.3416,1.4142,0.1577
8,arima_aic,20,all_origins,750,1.3620,0.1736,1.0951,0.2738
9,arima_aic,20,converged_only,487,1.0461,0.2960,1.0385,0.2995


In [13]:
# Convergence summary, derived from the flags already captured inline in the
# main forecasting loop above (issue #71) -- no longer re-fits every origin a
# second time just to check convergence (previously ~1,400 duplicate fits).

convergence_parts = []
for model_name, order in ARIMA_ORDERS.items():
    per_origin = results_df.drop_duplicates(subset=["origin_idx"])[
        ["origin_idx", "origin_date", f"{model_name}_converged"]
    ].rename(columns={f"{model_name}_converged": "converged"})
    per_origin["model"] = model_name
    per_origin["order"] = str(order)
    convergence_parts.append(per_origin)

convergence_df = pd.concat(convergence_parts, ignore_index=True)

convergence_summary = (
    convergence_df
    .groupby(["model", "order"])["converged"]
    .agg(
        total_fits="count",
        converged_fits="sum"
    )
    .reset_index()
)

convergence_summary["non_converged_fits"] = (
    convergence_summary["total_fits"]
    - convergence_summary["converged_fits"]
)

convergence_summary["convergence_rate_pct"] = (
    convergence_summary["converged_fits"]
    / convergence_summary["total_fits"]
    * 100
)

convergence_summary

,model,order,total_fits,converged_fits,non_converged_fits,convergence_rate_pct
0,arima_aic,"(2, 0, 3)",750,487,263,64.933333
1,arima_bic,"(0, 0, 0)",750,717,33,95.600000


In [14]:
# Retry only non-converged expanding-window fits with a higher iteration limit

retry_records = []

failed_fits = convergence_df[
    convergence_df["converged"] == False
].copy()

for _, failed in failed_fits.iterrows():

    origin = int(failed["origin_idx"])
    model_name = failed["model"]
    order = ARIMA_ORDERS[model_name]

    train_diff = arima_diff.iloc[:origin - 1][TARGET].astype(float)

    retry_fit = ARIMA(
        train_diff,
        order=order,
        trend="c"
    ).fit(
        method_kwargs={"maxiter": 500}
    )

    retry_records.append({
        "origin_idx": origin,
        "origin_date": arima_levels.loc[origin - 1, "date"],
        "model": model_name,
        "order": str(order),
        "retry_converged": bool(
            retry_fit.mle_retvals.get("converged", True)
        ),
    })

retry_df = pd.DataFrame(retry_records)

retry_summary = (
    retry_df
    .groupby("model")["retry_converged"]
    .agg(
        retried="count",
        converged_after_retry="sum"
    )
    .reset_index()
)

retry_summary["still_not_converged"] = (
    retry_summary["retried"]
    - retry_summary["converged_after_retry"]
)

retry_summary

,model,retried,converged_after_retry,still_not_converged
0,arima_aic,263,246,17
1,arima_bic,33,0,33


### Expanding-window convergence diagnostic

Because ARIMA models are re-estimated at every expanding-window forecast origin, convergence was
checked for each fitted specification -- captured inline, at the point each model is actually
fit in the main forecasting loop above, rather than by re-fitting every origin a second time just
to read this flag (issue #71).

| Model | Order | Total fits | Converged | Non-converged | Convergence rate |
|---|---|---:|---:|---:|---:|

*(table populated by the cell above -- see `convergence_summary`)*

A retry of all non-converged fits with a higher maximum iteration limit (`maxiter=500`) recovers
most of ARIMA-AIC's (246 of 263, 94%) but none of ARIMA-BIC's (0 of 33) -- see `retry_summary`
below.

**This is no longer just a documented caveat: `arima_aic`/`arima_bic`'s reported RMSE/MAE and
Diebold-Mariano results above are now given both unconditionally (`all_origins`, matching this
notebook's original behavior) and restricted to origins where that model's own fit actually
converged (`converged_only`)** -- so the non-convergent fits' effect on the headline numbers is
visible rather than assumed away. All forecast origins still produced finite predictions with no
missing or infinite values, and forecast ranges remained consistent with the observed yield-spread
range regardless of convergence status.

In [15]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_df.to_csv(
    OUTPUT_DIR / "r3_arima_vs_naive.csv",
    index=False
)

results_df.to_csv(
    OUTPUT_DIR / "r3_arima_forecasts.csv",
    index=False
)

dm_results_df.to_csv(
    OUTPUT_DIR / "r3_arima_diebold_mariano.csv",
    index=False
)

convergence_summary.to_csv(
    OUTPUT_DIR / "r3_arima_convergence.csv",
    index=False
)

# Convergence-conditional metrics (issue #71) -- new file, doesn't overwrite
# or replace the unconditional r3_arima_vs_naive.csv above.
metrics_by_convergence_df.to_csv(
    OUTPUT_DIR / "r3_arima_metrics_by_convergence.csv",
    index=False
)

print("Saved ARIMA outputs:")
print("- r3_arima_vs_naive.csv")
print("- r3_arima_forecasts.csv (now includes arima_aic_converged / arima_bic_converged)")
print("- r3_arima_diebold_mariano.csv (now includes an all_origins/converged_only 'sample' column)")
print("- r3_arima_convergence.csv")
print("- r3_arima_metrics_by_convergence.csv")

Saved ARIMA outputs:
- r3_arima_vs_naive.csv
- r3_arima_forecasts.csv (now includes arima_aic_converged / arima_bic_converged)
- r3_arima_diebold_mariano.csv (now includes an all_origins/converged_only 'sample' column)
- r3_arima_convergence.csv
- r3_arima_metrics_by_convergence.csv


## ARIMA Baseline Conclusion

The ARIMA baseline was implemented on the stationary first-differenced Canadian 10Y-2Y yield-spread series using the same processed data and evaluation window as the VAR baselines in `#28` and `#29`.

To preserve direct comparability, the ARIMA evaluation used the same common sample:

- **4,268 complete level observations, 4,267 differenced modeling observations** -- as of issue #63, this notebook reads `data/processed/gold_features.csv` directly instead of independently rebuilding an inner-joined, no-fill frame that silently dropped 259 genuine trading days VECM/LSTM's Gold-layer pipeline kept, and the expanding-window loop's `origin` counts *level* observations trained on (matching `evaluate_vecm()`'s convention) rather than *differenced* observations.
- Expanding-window evaluation
- Initial training sample: **500 observations**
- Evaluation step: **5 observations**
- Forecast horizons: **1, 5, and 20 days**
- Shared benchmark: **Random Walk**

### Order selection

An automated grid search over ARIMA specifications with `p, q ∈ {0, ..., 5}` was performed on the stationary differenced yield-spread series.

The selected specifications were:

- **AIC-selected model:** `ARIMA(2, 0, 3)`
  - AIC = `-18143.900`
  - BIC = `-18099.389`

- **BIC-selected model:** `ARIMA(0, 0, 0)`
  - AIC = `-18131.073`
  - BIC = `-18118.356`

Both full-sample specifications converged successfully.

Because the yield spread was first-differenced prior to model estimation, `d = 0` in these ARIMA specifications. The differencing required to achieve stationarity was therefore applied explicitly before fitting the models.

### Forecast performance (all origins)

| Horizon | ARIMA-AIC RMSE | ARIMA-AIC MAE | ARIMA-BIC RMSE | ARIMA-BIC MAE | Random Walk RMSE | Random Walk MAE |
|---|---:|---:|---:|---:|---:|---:|
| 1 day | 0.029104 | 0.020794 | 0.029010 | 0.020665 | 0.029001 | 0.020587 |
| 5 days | 0.061872 | 0.047193 | 0.061355 | 0.046630 | 0.061161 | 0.046333 |
| 20 days | 0.128153 | 0.096310 | 0.126546 | 0.095691 | 0.125057 | 0.094387 |

Neither ARIMA specification outperformed the Random Walk benchmark across the three forecast horizons.

Relative to the Random Walk:

- **ARIMA-AIC**
  - 1 day: RMSE +0.36%, MAE +1.01%
  - 5 days: RMSE +1.16%, MAE +1.86%
  - 20 days: RMSE +2.48%, MAE +2.04%

- **ARIMA-BIC**
  - 1 day: RMSE +0.03%, MAE +0.38%
  - 5 days: RMSE +0.32%, MAE +0.64%
  - 20 days: RMSE +1.19%, MAE +1.38%

The BIC-selected specification remained closer to the Random Walk than the AIC-selected specification, but still did not improve forecast accuracy.

### Diebold-Mariano significance (all origins)

The Diebold-Mariano test was applied using horizon-specific truncation lags (`h = 1, 5, 20`) to account for the serial dependence induced by multi-step forecast errors.

For **ARIMA-AIC vs Random Walk**:

- **1 day:** Random Walk significantly better under absolute-error loss (`p = 0.0160`); squared-error difference not significant (`p = 0.2253`).
- **5 days:** Random Walk significantly better under both squared-error (`p = 0.0463`) and absolute-error (`p = 0.0287`) loss.
- **20 days:** no statistically significant difference under either squared-error (`p = 0.1736`) or absolute-error (`p = 0.2738`) loss.

For **ARIMA-BIC vs Random Walk**:

- **1 day:** Random Walk significantly better under absolute-error loss (`p = 0.0239`); squared-error difference not significant (`p = 0.7905`).
- **5 days:** no statistically significant difference under either squared-error (`p = 0.2272`) or absolute-error (`p = 0.0869`) loss.
- **20 days:** no statistically significant difference under either squared-error (`p = 0.2750`) or absolute-error (`p = 0.2750`) loss.

Both ARIMA specifications have consistently higher RMSE and MAE than the Random Walk at all horizons.

### Convergence diagnostic

Because the ARIMA models were re-estimated at every expanding-window forecast origin, optimizer convergence was monitored across all 750 fits per specification.

- `ARIMA(2, 0, 3)`: **487 / 750 converged (64.9%)**
- `ARIMA(0, 0, 0)`: **717 / 750 converged (95.6%)**

The AIC-selected specification's convergence rate is markedly worse than the BIC-selected one: mixed AR+MA orders are more prone to near-unidentifiable AR/MA root cancellation during MLE than a pure-MA or pure-drift order. **Unlike a documented-but-inert caveat, retrying non-converged fits with a higher iteration limit (`maxiter=500`) recovers most of them for AIC -- 246 of 263 (94%) converge on retry, leaving 17 genuinely non-convergent.** BIC's 33 non-converged fits are not recovered by the same retry (0%) -- its drift-model specification's rare convergence failures appear to be genuine degenerate cases, not just under-iterated ones.

Despite the non-convergent fits, all forecast origins produced finite predictions with no missing or infinite values, and forecast ranges remained consistent with the observed yield-spread range.

### Convergence-conditional results (issue #71)

Rather than leaving convergence as a documented-but-unaddressed caveat, `results_df` now carries
each origin's convergence flag inline (`arima_aic_converged`, `arima_bic_converged`), and every
RMSE/MAE/Diebold-Mariano result above is computed both **unconditionally** (`all_origins`,
identical methodology to the tables above) and **restricted to origins where that model's own fit
converged** (`converged_only`) -- see `metrics_by_convergence_df` and `dm_results_df`'s `sample`
column.

**Result: restricting BIC to converged-only origins does not change the finding** -- BIC still
underperforms Random Walk on RMSE/MAE at every horizon in both samples, and no BIC Diebold-Mariano
verdict flips across the p=0.05 threshold between samples.

**AIC is different, and this is the substantive question issue #71 asked.** With only 64.9% of
AIC fits converging, restricting to `converged_only` (487 of 750 origins) is not a small change to
the sample, and it changes the statistical conclusion, not just the numbers: at h=1, Random Walk's
absolute-error advantage over ARIMA-AIC is significant on the full sample (`p = 0.0160`) but **not**
on the converged-only sample (`p = 0.0507`, just over the threshold). At h=5, Random Walk's
advantage under **both** squared-error (`p = 0.0463` -> `0.3374`) and absolute-error
(`p = 0.0287` -> `0.1529`) loss is significant on the full sample but not on converged-only. In
both cases the RMSE/MAE point estimates stay in the same direction (ARIMA-AIC still numerically
worse than naive on converged-only origins too -- e.g. h=5 RMSE degrades from -1.16% to -0.69% but
is still negative) -- restricting to converged fits narrows the gap and removes its statistical
significance, it does not reverse it. h=20 shows no significant difference in either sample. The
non-convergent origins are not random noise as far as this test can tell: they carry some of the
signal behind ARIMA-AIC's headline underperformance at h=1/h=5, which is exactly why gating on
convergence (this issue) matters more here than it would have on the pre-#63 sample, where the
AIC-selected order converged at 93%+ and no flips occurred.

### Overall finding

The ARIMA statistical baseline does not outperform the Random Walk benchmark for Canadian 10Y-2Y yield-spread forecasting under the shared 1-, 5-, and 20-day evaluation framework, on either the full or the convergence-restricted sample.

AIC favored a mixed `ARMA(2,3)` structure, while BIC favored the parsimonious `ARIMA(0,0,0)` specification. Neither information-criterion choice produced superior predictive accuracy relative to the naive benchmark. The AIC-selected specification's much lower convergence rate (65% vs BIC's 96%) is not just a documentation caveat here -- restricting to converged-only fits measurably weakens (though does not reverse) ARIMA-AIC's apparent underperformance at the 1- and 5-day horizons, so results built on this specification should report both samples rather than the unconditional one alone.

These results provide the required univariate statistical baseline for subsequent comparison with VAR/VECM and LSTM models.